## 0. Create Spark session

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark import SparkContext

import pandas as pd
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)
def show_df(input_df):
    input_df.printSchema()
    print(f"COUNT = {input_df.count()}")
    return input_df.limit(10).toPandas()

os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.hadoop:hadoop-aws:3.2.0,org.apache.hadoop:hadoop-common:3.2.0 '
    'pyspark-shell'
)
os.environ['S3_ENDPOINT'] = "http://minio:9000"
os.environ['AWS_ACCESS_KEY_ID'] = "minio"
os.environ['AWS_SECRET_ACCESS_KEY'] = "minio123"

if 'spark' in globals():
    spark.stop()
    print("Spark session stopped.")
else:
    print("No Spark session to stop.")

spark = (
    SparkSession.builder#.master("spark://spark:7077")
    .appName("notebook-velib-to-siler")
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("AWS_ACCESS_KEY_ID"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("AWS_SECRET_ACCESS_KEY"))
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("S3_ENDPOINT"))
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "1")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000")
    .config("spark.hadoop.fs.s3a.connection.timeout", "10000")
    .getOrCreate()
)


os.environ['PART_DAY'] = '2024-10-30'
os.environ['S3_INPUT_PATH'] = f"s3a://velib/silver/velib-disponibilite-en-temps-reel"
os.environ['S3_OUTPUT_PATH'] = "s3a://velib/gold"

No Spark session to stop.


:: loading settings :: url = jar:file:/usr/local/spark-3.1.2-bin-hadoop3.2/jars/ivy-2.4.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.hadoop#hadoop-common added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d684b2b8-8819-40f4-945f-22df70f4e5a7;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.2.0 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.375 in central
	found org.apache.hadoop#hadoop-common;3.2.0 in central
	found org.apache.hadoop#hadoop-annotations;3.2.0 in central
	found com.google.guava#guava;11.0.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found commons-cli#commons-cli;1.2 in central
	found org.apache.commons#commons-math3;3.1.1 in central
	found org.apache.httpcomponents#httpclient;4.5.2 in central
	found org.apache.httpcomponents#httpcore;4.4.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.11 in c

## 1. Extract from object storage

In [2]:
spark.sparkContext.setLogLevel("WARN")

df = spark.read.parquet(f"{os.getenv('S3_INPUT_PATH')}/part_day={os.environ['PART_DAY']}").cache()
show_df(df)

24/11/02 23:09:30 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- capacity: long (nullable = true)
 |-- code_insee_commune: string (nullable = true)
 |-- duedate: string (nullable = true)
 |-- duedate_timestamp_minute: timestamp (nullable = true)
 |-- ebike: long (nullable = true)
 |-- fill_percentage: double (nullable = true)
 |-- fill_ratio: double (nullable = true)
 |-- is_installed: string (nullable = true)
 |-- is_renting: string (nullable = true)
 |-- is_returning: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- mechanical: long (nullable = true)
 |-- name: string (nullable = true)
 |-- nom_arrondissement_communes: string (nullable = true)
 |-- numbikesavailable: long (nullable = true)
 |-- numdocksavailable: long (nullable = true)
 |-- numero_departement: string (nullable = true)
 |-- part_day: string (nullable = true)
 |-- part_minute: string (nullable = true)
 |-- part_month: string (nullable = true)
 |-- polldate: string (nullable = true)
 |-- polldate_timestamp_minute: timestamp

COUNT = 1356405


,capacity,code_insee_commune,duedate,duedate_timestamp_minute,ebike,fill_percentage,fill_ratio,is_installed,is_renting,is_returning,lat,lon,mechanical,name,nom_arrondissement_communes,numbikesavailable,numdocksavailable,numero_departement,part_day,part_minute,part_month,polldate,polldate_timestamp_minute,stationcode
0,35,75056,2024-10-30T12:13:36+00:00,2024-10-30 12:13:00,3,17.1,0.171,OUI,OUI,OUI,48.865983,2.275725,3,Benjamin Godard - Victor Hugo,Paris,6,29,75,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,16107
1,21,75056,2024-10-30T12:05:31+00:00,2024-10-30 12:05:00,0,0.0,0.000,OUI,OUI,OUI,48.879296,2.337360,0,Toudouze - Clauzel,Paris,0,21,75,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,9020
2,20,94081,2024-10-30T12:13:35+00:00,2024-10-30 12:13:00,3,55.0,0.550,OUI,OUI,OUI,48.778193,2.396302,8,Rouget de L'isle - Watteau,Vitry-sur-Seine,11,9,94,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,44015
3,21,75056,2024-10-30T12:09:30+00:00,2024-10-30 12:09:00,2,42.9,0.429,OUI,OUI,OUI,48.851654,2.330808,7,Saint-Sulpice,Paris,9,9,75,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,6003
4,25,75056,2024-10-30T12:10:59+00:00,2024-10-30 12:10:00,2,12.0,0.120,OUI,OUI,OUI,48.837526,2.336035,1,Cassini - Denfert-Rochereau,Paris,3,18,75,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,14111
5,23,75056,2024-10-30T12:11:41+00:00,2024-10-30 12:11:00,1,4.3,0.043,OUI,OUI,OUI,48.843893,2.351966,0,Lacépède - Monge,Paris,1,22,75,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,5110
6,60,75056,2024-10-30T12:13:43+00:00,2024-10-30 12:13:00,7,35.0,0.350,OUI,OUI,OUI,48.819428,2.343335,14,Jourdan - Stade Charléty,Paris,21,38,75,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,14014
7,17,75056,2024-10-30T12:11:14+00:00,2024-10-30 12:11:00,7,82.4,0.824,OUI,OUI,OUI,48.847082,2.321375,7,Saint-Romain - Cherche-Midi,Paris,14,0,75,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,6108
8,22,93066,2024-10-30T12:09:28+00:00,2024-10-30 12:09:00,6,36.4,0.364,OUI,OUI,OUI,48.936269,2.358867,2,Basilique,Saint-Denis,8,13,93,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,32017
9,31,93001,2024-10-30T12:07:02+00:00,2024-10-30 12:07:00,4,38.7,0.387,OUI,OUI,OUI,48.910399,2.385136,8,André Karman - République,Aubervilliers,12,17,93,2024-10-30,2024-10-30T12:34,2024-10,2024-10-30T12:34:00+00:00,2024-10-30 12:34:00,33006


## 2. Transform

In [3]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

def calc_turnover_rate_by_station_10min(input_df):
    windowSpec  = Window.partitionBy("stationcode", "ten_minute_interval").orderBy("polldate_timestamp_minute")

    df_transformed = input_df#.dropDuplicates(["stationcode", "polldate_timestamp_minute"])
    df_transformed = (
        df_transformed
        .withColumn("ten_minute_interval", F.substring("polldate", 1, 15))
        .withColumn("numbikesavailable_abs_lag_diff",
            F.abs(F.col("numbikesavailable") - F.lag("numbikesavailable", 1).over(windowSpec)))
        )
    df_transformed = (
        df_transformed
        .groupBy("stationcode", "ten_minute_interval")
        .agg(
            F.sum(F.col("numbikesavailable_abs_lag_diff")).alias("turnover_rate_10min"),
            F.first(F.col("name")).alias("name"),
            F.first(F.col("lat")).alias("lat"),
            F.first(F.col("lon")).alias("lon"),
            F.first(F.col("capacity")).alias("capacity"),
            F.first(F.col("part_day")).alias("part_day"),
            F.first(F.col("nom_arrondissement_communes")).alias("nom_arrondissement_communes"),
            F.first(F.col("code_insee_commune")).alias("code_insee_commune"),
            F.first(F.col("numero_departement")).alias("numero_departement"),
            F.first(F.col("polldate_timestamp_minute")).alias("polldate_timestamp_minute")
        )
        .filter(F.col("turnover_rate_10min") < 500)
        .na.fill({'turnover_rate_10min': .0})
    )
    df_transformed = df_transformed.select(*(sorted(df_transformed.columns)))
    return df_transformed

def calc_summary_stats_by_station_1hour(input_df):
    df_transformed = input_df
    df_transformed = (
        df_transformed
        .withColumn("one_hour_interval", F.substring("polldate", 1, 13))
        .groupBy("stationcode", "one_hour_interval")
        .agg(
            F.avg(F.col("numbikesavailable")).alias("numbikesavailable_avg_1hour"),
            F.min(F.col("numbikesavailable")).alias("numbikesavailable_min_1hour"),
            F.max(F.col("numbikesavailable")).alias("numbikesavailable_max_1hour"),
            F.avg(F.col("fill_ratio")).alias("fill_ratio_avg_1hour"),
            F.min(F.col("fill_ratio")).alias("fill_ratio_min_1hour"),
            F.max(F.col("fill_ratio")).alias("fill_ratio_max_1hour"),
            F.first(F.col("name")).alias("name"),
            F.first(F.col("lat")).alias("lat"),
            F.first(F.col("lon")).alias("lon"),
            F.first(F.col("capacity")).alias("capacity"),
            F.first(F.col("part_day")).alias("part_day"),
            F.first(F.col("nom_arrondissement_communes")).alias("nom_arrondissement_communes"),
            F.first(F.col("code_insee_commune")).alias("code_insee_commune"),
            F.first(F.col("numero_departement")).alias("numero_departement"),
            F.first(F.col("polldate_timestamp_minute")).alias("polldate_timestamp_minute")
        )
        .na.fill({
            'numbikesavailable_avg_1hour': .0,
            'numbikesavailable_min_1hour': .0,
            'numbikesavailable_max_1hour': .0,
            'fill_ratio_avg_1hour': .0,
            'fill_ratio_min_1hour': .0,
            'fill_ratio_max_1hour': .0
        })
    )
    df_transformed = df_transformed.select(*(sorted(df_transformed.columns)))
    return df_transformed


def calc_summary_stats_1hour(input_df):
    df_transformed = input_df
    df_transformed = (
        df_transformed
        .withColumn("one_hour_interval", F.substring("polldate", 1, 13))
        .groupBy("one_hour_interval")
        .agg(
            F.avg(F.col("numbikesavailable")).alias("numbikesavailable_avg_1hour"),
            F.min(F.col("numbikesavailable")).alias("numbikesavailable_min_1hour"),
            F.max(F.col("numbikesavailable")).alias("numbikesavailable_max_1hour"),
            F.avg(F.col("fill_ratio")).alias("fill_ratio_avg_1hour"),
            F.min(F.col("fill_ratio")).alias("fill_ratio_min_1hour"),
            F.max(F.col("fill_ratio")).alias("fill_ratio_max_1hour"),
            F.first(F.col("name")).alias("name"),
            F.first(F.col("lat")).alias("lat"),
            F.first(F.col("lon")).alias("lon"),
            F.first(F.col("capacity")).alias("capacity"),
            F.first(F.col("part_day")).alias("part_day"),
            F.first(F.col("nom_arrondissement_communes")).alias("nom_arrondissement_communes"),
            F.first(F.col("code_insee_commune")).alias("code_insee_commune"),
            F.first(F.col("numero_departement")).alias("numero_departement"),
            F.first(F.col("polldate_timestamp_minute")).alias("polldate_timestamp_minute")
        )
        .na.fill({
            'numbikesavailable_avg_1hour': .0,
            'numbikesavailable_min_1hour': .0,
            'numbikesavailable_max_1hour': .0,
            'fill_ratio_avg_1hour': .0,
            'fill_ratio_min_1hour': .0,
            'fill_ratio_max_1hour': .0
        })
    )
    df_transformed = df_transformed.select(*(sorted(df_transformed.columns)))
    return df_transformed

## 3. Load into object storage

In [4]:
def write_parquet(input_df, dataset_name):
    (
        input_df.write
        .format("parquet")
        .mode("overwrite")
        .save(f"{os.getenv('S3_OUTPUT_PATH')}/{dataset_name}/part_day={os.getenv('PART_DAY')}")
    )
    
# write_parquet(df_dict[dataset_name], dataset_name)

## 4. Update hive metastore

In [5]:
!pip install trino

In [6]:
from pyspark.sql import DataFrame

def generate_trino_create_table(
    df: DataFrame, 
    catalog: str, 
    schema: str, 
    table: str, 
    partitioned_by: str, 
    external_location: str
) -> str:
    # Comprehensive mapping of Spark SQL types to Trino types
    type_mapping = {
        "bigint": "BIGINT",
        "binary": "VARBINARY",
        "boolean": "BOOLEAN",
        "decimal": "DECIMAL",  # Precision and scale will need to be handled separately if defined
        "double": "DOUBLE",
        "float": "REAL",
        "int": "INTEGER",
        "smallint": "SMALLINT",
        "string": "VARCHAR",
        "timestamp": "TIMESTAMP",
        "tinyint": "TINYINT"
    }
    
    # Separate columns to ensure partition column is placed last
    columns = []
    partition_column = None
    for field in df.schema.fields:
        spark_type = field.dataType.simpleString()
        trino_type = type_mapping.get(spark_type, "VARCHAR")  # default to VARCHAR if no match
        # DEBUG # print(f'field.name={field.name}, spark_type={spark_type}, trino_type={trino_type}')
        if field.name == partitioned_by:
            partition_column = f"{field.name} {trino_type}"
        else:
            columns.append(f"{field.name} {trino_type}")
    
    # Add the partition column to the end if it exists in the schema
    if partition_column:
        columns.append(partition_column)
    
    # Join column definitions into a single string
    columns_definition = ",\n    ".join(columns)
    
    # Generate the final CREATE TABLE statement
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
        {columns_definition}
    ) 
    WITH (
        format = 'PARQUET',
        partitioned_by = ARRAY['{partitioned_by}'],
        external_location = '{external_location}'
    )
    """
    
    return create_table_sql.strip()


In [7]:
import trino

def connect_to_trino(host='trino-coordinator', port=8080, user='trino'):
    return trino.dbapi.connect(host=host, port=port, user=user)

def create_table(input_df, dataset_name):
    conn = connect_to_trino()
    cur = conn.cursor()

    catalog, schema, table = 'minio', 'velib_gold', dataset_name
    partitioned_by = 'part_day'
    schema_location = os.environ["S3_OUTPUT_PATH"]
    external_location = f'{os.environ["S3_OUTPUT_PATH"]}/{dataset_name}'

    # List of queries to execute
    queries = [
        f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema} WITH (location = '{schema_location}')",
        f"DROP TABLE IF EXISTS {catalog}.{schema}.{table}",

        generate_trino_create_table(input_df, catalog, schema, table, partitioned_by, external_location),

        f"USE {catalog}.{schema}",
        f"CALL system.sync_partition_metadata('{schema}', '{table}', 'ADD')",
        f"SELECT * FROM {catalog}.{schema}.{table} LIMIT 5"
    ]

    # Execute each query in the list
    for query in queries:
        try:
            cur.execute(query)
            # Check if the query is a SELECT query to fetch results
            if query.startswith("SELECT"):
                results = cur.fetchall()
                for row in results:
                    print(row)
            else:
                print(f"Executed: {query}")
        except Exception as e:
            raise Exception(f"Error executing query: {query}. Error: {e}")

    # Close the cursor and connection
    cur.close()
    conn.close()


## Run functions

In [8]:
dataset_transform_function_map = {
    'turnover_rate_by_station_10min': calc_turnover_rate_by_station_10min,
    'summary_stats_by_station_1hour': calc_summary_stats_by_station_1hour,
    'summary_stats_1hour': calc_summary_stats_1hour
}

for dataset_name, transform_function in dataset_transform_function_map.items():
    df_transformed = transform_function(df)
    write_parquet(df_transformed, dataset_name)
    create_table(df_transformed, dataset_name)

24/11/02 23:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/11/02 23:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
24/11/02 23:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
24/11/02 23:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
24/11/02 23:09:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
24/11/02 23:09:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
24/11/02 23:09:48 WARN MemoryManager: Total allocation exceeds 95.

Executed: CREATE SCHEMA IF NOT EXISTS minio.velib_gold WITH (location = 's3a://velib/gold')
Executed: DROP TABLE IF EXISTS minio.velib_gold.turnover_rate_by_station_10min
Executed: CREATE TABLE IF NOT EXISTS minio.velib_gold.turnover_rate_by_station_10min (
        capacity BIGINT,
    code_insee_commune VARCHAR,
    lat DOUBLE,
    lon DOUBLE,
    name VARCHAR,
    nom_arrondissement_communes VARCHAR,
    numero_departement VARCHAR,
    polldate_timestamp_minute TIMESTAMP,
    stationcode VARCHAR,
    ten_minute_interval VARCHAR,
    turnover_rate_10min BIGINT,
    part_day VARCHAR
    ) 
    WITH (
        format = 'PARQUET',
        partitioned_by = ARRAY['part_day'],
        external_location = 's3a://velib/gold/turnover_rate_by_station_10min'
    )
Executed: USE minio.velib_gold
Executed: CALL system.sync_partition_metadata('velib_gold', 'turnover_rate_by_station_10min', 'ADD')
[19, '75056', 48.872089449907, 2.3575825989246, 'Mairie du 10ème', 'Paris', '75', datetime.datetime(2024

24/11/02 23:10:09 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
24/11/02 23:10:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/11/02 23:10:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
24/11/02 23:10:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
24/11/02 23:10:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
24/11/02 23:10:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
24/11/02 23:10:12 WARN MemoryManager: Tota

Executed: CREATE SCHEMA IF NOT EXISTS minio.velib_gold WITH (location = 's3a://velib/gold')
Executed: DROP TABLE IF EXISTS minio.velib_gold.summary_stats_by_station_1hour
Executed: CREATE TABLE IF NOT EXISTS minio.velib_gold.summary_stats_by_station_1hour (
        capacity BIGINT,
    code_insee_commune VARCHAR,
    fill_ratio_avg_1hour DOUBLE,
    fill_ratio_max_1hour DOUBLE,
    fill_ratio_min_1hour DOUBLE,
    lat DOUBLE,
    lon DOUBLE,
    name VARCHAR,
    nom_arrondissement_communes VARCHAR,
    numbikesavailable_avg_1hour DOUBLE,
    numbikesavailable_max_1hour BIGINT,
    numbikesavailable_min_1hour BIGINT,
    numero_departement VARCHAR,
    one_hour_interval VARCHAR,
    polldate_timestamp_minute TIMESTAMP,
    stationcode VARCHAR,
    part_day VARCHAR
    ) 
    WITH (
        format = 'PARQUET',
        partitioned_by = ARRAY['part_day'],
        external_location = 's3a://velib/gold/summary_stats_by_station_1hour'
    )
Executed: USE minio.velib_gold
Executed: CALL syste

24/11/02 23:10:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/11/02 23:10:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
24/11/02 23:10:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
24/11/02 23:10:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
24/11/02 23:10:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
24/11/02 23:10:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
24/11/02 23:10:32 WARN MemoryManager: Total allocation exceeds 95.

Executed: CREATE SCHEMA IF NOT EXISTS minio.velib_gold WITH (location = 's3a://velib/gold')
Executed: DROP TABLE IF EXISTS minio.velib_gold.summary_stats_1hour
Executed: CREATE TABLE IF NOT EXISTS minio.velib_gold.summary_stats_1hour (
        capacity BIGINT,
    code_insee_commune VARCHAR,
    fill_ratio_avg_1hour DOUBLE,
    fill_ratio_max_1hour DOUBLE,
    fill_ratio_min_1hour DOUBLE,
    lat DOUBLE,
    lon DOUBLE,
    name VARCHAR,
    nom_arrondissement_communes VARCHAR,
    numbikesavailable_avg_1hour DOUBLE,
    numbikesavailable_max_1hour BIGINT,
    numbikesavailable_min_1hour BIGINT,
    numero_departement VARCHAR,
    one_hour_interval VARCHAR,
    polldate_timestamp_minute TIMESTAMP,
    part_day VARCHAR
    ) 
    WITH (
        format = 'PARQUET',
        partitioned_by = ARRAY['part_day'],
        external_location = 's3a://velib/gold/summary_stats_1hour'
    )
Executed: USE minio.velib_gold
Executed: CALL system.sync_partition_metadata('velib_gold', 'summary_stats_1ho